# EV 화재 FDS 결과 후처리 — 브리프 8장 검증 체크리스트`input/build_fds_input.ipynb`가 만든 케이스의 결과를 읽어| # | 항목 | 이 노트북에서 ||---|---|---|| 1 | 에너지 보존 | `HRR_BATT` 적분 vs 1D THR || 2 | 케이싱 열유속 역산 | `q_PACK_*` → `feedback_1d/` CSV 내보내기 (브리프 5.2 3~4단계) || 3 | HRR 곡선 형상 | 피크 시점·상승 기울기 || 4 | 총 THR | 차체 `THICKNESS` 튜닝 근거 || 5 | 격자 민감도 | `EV_dx100 / EV_demo / EV_dx200` || 6 | 복사분율 민감도 | `EV_chi010 / EV_demo / EV_chi025` || 7 | 미연소 가스 | `BATTGAS_underbody` |를 확인한다. 5·6번은 보고서 필수 항목.

## 0. 로드

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT     = Path("/scratch/x3319a05/FDS/electric_vehicle_battery")
RESULTS  = ROOT / "results"
DATA_1D  = ROOT / "input" / "data_1d"
FIGS     = RESULTS / "figures"
FEEDBACK = RESULTS / "feedback_1d"
for d in (FIGS, FEEDBACK):
    d.mkdir(parents=True, exist_ok=True)

# 이 클러스터에는 한글 폰트가 없다. 그림 안의 글자는 전부 영문으로 쓰고,
# 한글 설명은 마크다운/표준출력에만 둔다 (그림에 두면 두부글자가 된다).
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 140, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3, "figure.autolayout": True})
_trapz = getattr(np, "trapezoid", None) or np.trapz


def read_fds_csv(path: Path) -> pd.DataFrame:
    """FDS CSV: 1행 단위, 2행 헤더, 3행부터 데이터."""
    units = pd.read_csv(path, nrows=1).columns.tolist()
    df = pd.read_csv(path, skiprows=1)
    df.attrs["units"] = dict(zip(df.columns, units))
    return df


def load_case(chid: str) -> dict | None:
    d = RESULTS / chid
    hrr = d / f"{chid}_hrr.csv"
    if not hrr.exists():
        return None
    out = {"chid": chid, "dir": d, "hrr": read_fds_csv(hrr)}
    devc = d / f"{chid}_devc.csv"
    if devc.exists():
        out["devc"] = read_fds_csv(devc)
    out_file = d / f"{chid}.out"
    out["done"] = out_file.exists() and "successfully" in out_file.read_text(errors="ignore")[-4000:]
    return out


ALL = ["EV_probe", "EV_quick", "EV_demo", "EV_dx100", "EV_dx200", "EV_chi010", "EV_chi025"]
C = {k: v for k in ALL if (v := load_case(k)) is not None}
print("가용 케이스:", {k: ("완료" if v["done"] else "진행중/중단") for k, v in C.items()})

# 주 케이스 우선순위: 완주한 본 케이스 > 스모크 테스트 > 프로브
for _p in ("EV_demo", "EV_quick", "EV_probe"):
    if _p in C:
        MAIN = _p; break
else:
    MAIN = list(C)[0] if C else None
print("주 케이스:", MAIN)

## 1D 기준값`input/data_1d/`의 1D 산출물을 그대로 읽어 FDS 결과와 비교할 기준선을 만든다.

In [ ]:
Z1D = [pd.read_csv(DATA_1D / f"vent_zone{i}.csv") for i in (1, 2, 3)]
CAS1D = pd.read_csv(DATA_1D / "casing_heat_flux.csv")
THR1D = pd.read_csv(DATA_1D / "thr_1d.csv")

MW  = {"H2": 2.016, "CO": 28.010, "CO2": 44.010, "CH4": 16.043, "C2H4": 28.054}
DHC = {"H2": 120.0, "CO": 10.1, "CO2": 0.0, "CH4": 50.0, "C2H4": 47.2}
_x = {"H2": .28, "CO": .23, "CO2": .28, "CH4": .12, "C2H4": .09}
_wmix = sum(_x[k] * MW[k] for k in _x)
DHC_MIX = sum(_x[k] * MW[k] / _wmix * DHC[k] for k in _x) * 1000.0   # [kJ/kg]

t1d = Z1D[0].t_s.values
mdot_1d = sum(z.mdot_kg_s.values for z in Z1D)          # [kg/s]  전 존 합
hrr_1d = mdot_1d * DHC_MIX                              # [kW]    완전연소 가정 상한
thr_1d = np.concatenate([[0], np.cumsum(0.5 * (hrr_1d[1:] + hrr_1d[:-1]) * np.diff(t1d))]) / 1000.0

print(f"1D 총 방출질량 : {_trapz(mdot_1d, t1d):.2f} kg")
print(f"1D 배터리 HRR 피크 (완전연소 상한): {hrr_1d.max():.0f} kW")
print(f"1D 배터리 THR  : {thr_1d[-1]:.1f} MJ   (thr_1d.csv: {THR1D.THR_MJ.iloc[-1]:.1f} MJ)")

## 체크 #1 · #3 · #4 — HRR / THR`HRR_BATT`, `HRR_BODY`는 `QUANTITY='HRRPUV REAC'`의 전역 체적적분이라**반응별로 분리된 HRR**이다. 배터리 기여를 1D가 준 완전연소 상한과 비교하면미연소분 유무를 바로 알 수 있다 (브리프 8장 #1, #7).

In [ ]:
def hrr_figure(chid):
    c = C[chid]
    h, d = c["hrr"], c.get("devc")
    fig, ax = plt.subplots(2, 1, figsize=(8.5, 7), sharex=True)

    ax[0].plot(h["Time"], h["HRR"], lw=1.6, color="k", label="FDS total HRR")
    if d is not None and "HRR_BATT" in d:
        ax[0].plot(d["Time"], d["HRR_BATT"], lw=1.2, color="tab:red",  label="FDS battery (REAC=BATT)")
        ax[0].plot(d["Time"], d["HRR_BODY"], lw=1.2, color="tab:blue", label="FDS car body (REAC=BODY)")
    m = t1d <= h["Time"].max()
    ax[0].plot(t1d[m], hrr_1d[m], "--", lw=1.2, color="tab:orange",
               label="1D battery, complete-combustion upper bound")
    ax[0].set_ylabel("HRR [kW]")
    ax[0].set_title(f"{chid} - Heat Release Rate (brief ch.8 #1, #3)")
    ax[0].legend(fontsize=8)

    thr = np.concatenate([[0], np.cumsum(0.5 * (h["HRR"].values[1:] + h["HRR"].values[:-1])
                                         * np.diff(h["Time"].values))]) / 1e6
    ax[1].plot(h["Time"], thr, lw=1.6, color="k", label=f"FDS total THR ({thr[-1]:.2f} GJ)")
    ax[1].plot(t1d[m], thr_1d[m] / 1000.0, "--", lw=1.2, color="tab:orange",
               label=f"1D battery THR ({thr_1d[m][-1]/1000:.3f} GJ)")
    ax[1].axhspan(7.0, 8.0, color="tab:green", alpha=0.12,
                  label="Kang et al. BEV target 7-8 GJ")
    ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("THR [GJ]")
    ax[1].set_title("Total heat released (brief ch.8 #4 - THICKNESS tuning)")
    ax[1].legend(fontsize=8)

    fig.savefig(FIGS / f"{chid}_hrr_thr.png")
    return fig, thr[-1]


if MAIN:
    _, THR_MAIN = hrr_figure(MAIN)
    h = C[MAIN]["hrr"]
    i = int(h["HRR"].idxmax())
    print(f"[{MAIN}] 피크 HRR {h['HRR'].iloc[i]:.0f} kW @ t={h['Time'].iloc[i]:.0f} s, "
          f"총 THR {THR_MAIN:.2f} GJ")

## 체크 #2 — 케이싱 열유속 역산 및 1D 되먹임 (브리프 5.2)1D 팀은 팩 외부 표면 경계조건을 **가정**했다 (단열 / 대기 대류 / 고정 열유속).FDS는 차체 착화 후 팩이 실제로 받는 입사 열유속을 계산한다. 둘의 차이가 크면브리프 5.2의 1회 반복(1D 재실행)이 필요하다.아래 셀이 `results/feedback_1d/q_inc_pack_<CHID>.csv`를 내보낸다.**이 파일 한 개가 1D 입력파일의 경계조건 한 줄을 대체하는 것**이 "되먹임"의 실체다.

In [ ]:
PACK_FACES = ["PACK_BOT", "PACK_YMIN", "PACK_YMAX", "PACK_XMIN", "PACK_XMAX"]


def pack_feedback(chid):
    c = C[chid]
    d = c.get("devc")
    if d is None:
        print("devc 없음"); return None
    qcols = [f"q_{f}" for f in PACK_FACES if f"q_{f}" in d]
    Qcols = [f"Q_{f}" for f in PACK_FACES if f"Q_{f}" in d]
    Tcols = [f"T_{f}" for f in PACK_FACES if f"T_{f}" in d]

    fig, ax = plt.subplots(2, 1, figsize=(8.5, 7), sharex=True)
    for q in qcols:
        ax[0].plot(d["Time"], d[q], lw=1.1, label=q)
    m = CAS1D.t_s <= d["Time"].max()
    ax[0].plot(CAS1D.t_s[m], CAS1D.q_net_kW_m2[m], "k--", lw=1.4,
               label="1D assumed BC (NET_HEAT_FLUX input)")
    ax[0].set_ylabel("heat flux [kW/m2]")
    ax[0].set_title(f"{chid} - Pack surface incident flux: 1D assumption vs FDS (brief 5.2 step 3)")
    ax[0].legend(fontsize=7, ncol=2)

    for T in Tcols:
        ax[1].plot(d["Time"], d[T], lw=1.1, label=T)
    ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("wall temperature [C]")
    ax[1].legend(fontsize=7, ncol=2)
    fig.savefig(FIGS / f"{chid}_pack_flux.png")

    # --- 1D 되먹임용 CSV
    fb = pd.DataFrame({"t_s": d["Time"]})
    for q in qcols:
        fb[q.replace("q_", "q_inc_") + "_kW_m2"] = d[q]
    if Qcols:
        fb["Q_pack_total_kW"] = d[Qcols].sum(axis=1)
    fb["q_inc_area_weighted_kW_m2"] = d[qcols].mean(axis=1)
    p = FEEDBACK / f"q_inc_pack_{chid}.csv"
    fb.to_csv(p, index=False, float_format="%.6g")

    q_fds = float(d[qcols].mean(axis=1).max())
    q_1d = float(CAS1D.q_net_kW_m2.max())
    print(f"[{chid}] 팩 표면 피크 입사 열유속: FDS {q_fds:.2f} kW/m²  vs  1D 가정 {q_1d:.2f} kW/m²"
          f"  (비 {q_fds/max(q_1d,1e-9):.2f}x)")
    print(f"  -> {p.relative_to(ROOT)}  (1D 2차 해석 경계조건으로 투입)")
    if q_fds > 2 * q_1d:
        print("  !! 가정값의 2배 이상. 브리프 5.2의 1D 재실행(1회 반복)이 필요하다.")
    return fig


if MAIN:
    pack_feedback(MAIN)

## 체크 #7 — 미연소 벤트가스 축적 / 독성가스

In [ ]:
def gas_figure(chid):
    d = C[chid].get("devc")
    if d is None:
        return
    fig, ax = plt.subplots(3, 1, figsize=(8.5, 9), sharex=True)

    for col in ("T_plume_2m", "T_plume_3m", "T_plume_4m"):
        if col in d:
            ax[0].plot(d["Time"], d[col], lw=1.1, label=col)
    ax[0].set_ylabel("temperature [C]"); ax[0].legend(fontsize=8)
    ax[0].set_title(f"{chid} - Plume centreline temperature")

    if "BATTGAS_underbody" in d:
        ax[1].plot(d["Time"], d["BATTGAS_underbody"], lw=1.2, color="tab:purple")
    ax[1].set_ylabel("BATTERY_GAS mass fraction [-]")
    ax[1].set_title("Unburned vent gas under vehicle (brief ch.8 #7)")

    for col, c_ in (("O2_1p5m", "tab:green"), ("CO_1p5m", "tab:red"), ("HF_1p5m", "tab:brown")):
        if col in d:
            ax[2].plot(d["Time"], d[col], lw=1.1, color=c_, label=col)
    ax[2].set_yscale("log"); ax[2].set_xlabel("time [s]")
    ax[2].set_ylabel("volume fraction [-]"); ax[2].legend(fontsize=8)
    ax[2].set_title("Gas concentration 1.5 m beside vehicle (HF = inert tracer)")
    fig.savefig(FIGS / f"{chid}_gas.png")
    return fig


if MAIN:
    gas_figure(MAIN)

## 체크 #5 · #6 — 민감도 (보고서 필수)- **#5 격자**: 피크 HRR 변화 10% 이내여야 함- **#6 복사분율**: 팩→차체 복사가 착화 시점을 지배하므로 변화폭을 반드시 기록

In [ ]:
def ignition_time(c, thresh_kW=50.0):
    """차체 착화 시점 = HRR_BODY가 임계값을 처음 넘는 시각."""
    d = c.get("devc")
    if d is None or "HRR_BODY" not in d:
        return np.nan
    m = d["HRR_BODY"] > thresh_kW
    return float(d["Time"][m].min()) if m.any() else np.nan


def sensitivity(group, title, title_en, fname):
    have = [g for g in group if g in C]
    if len(have) < 2:
        print(f"[{title}] 케이스 부족 (가용: {have}) — 실행 후 다시 돌릴 것")
        return None
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    rows = []
    for g in have:
        h = C[g]["hrr"]
        ax.plot(h["Time"], h["HRR"], lw=1.3, label=g)
        thr = _trapz(h["HRR"], h["Time"]) / 1e6
        rows.append(dict(case=g, peak_HRR_kW=float(h["HRR"].max()),
                         t_peak_s=float(h["Time"][h["HRR"].idxmax()]),
                         THR_GJ=thr, t_body_ignite_s=ignition_time(C[g])))
    ax.set_xlabel("time [s]"); ax.set_ylabel("HRR [kW]"); ax.set_title(title_en)
    ax.legend(fontsize=8)
    fig.savefig(FIGS / fname)
    t = pd.DataFrame(rows).set_index("case")
    ref = t.index[len(t) // 2]
    t["peak_HRR_변화율_%"] = (t.peak_HRR_kW / t.loc[ref, "peak_HRR_kW"] - 1) * 100
    print(f"\n[{title}]  기준 = {ref}")
    print(t.to_string(float_format=lambda v: f"{v:10.2f}"))
    return t


T5 = sensitivity(["EV_dx100", "EV_demo", "EV_dx200"],
                 "격자 민감도 (브리프 8장 #5 — 피크 HRR 변화 10% 이내)",
                 "Grid sensitivity (brief ch.8 #5 - peak HRR within 10%)",
                 "sens_grid.png")
T6 = sensitivity(["EV_chi010", "EV_demo", "EV_chi025"],
                 "복사분율 민감도 (브리프 8장 #6 — 차체 착화 시점)",
                 "Radiative fraction sensitivity (brief ch.8 #6 - body ignition time)",
                 "sens_chi_r.png")

## 종합 판정표

In [ ]:
def verdict():
    if not MAIN:
        print("결과 없음"); return
    c = C[MAIN]; h = c["hrr"]; d = c.get("devc")
    rows = []

    # #1 에너지 보존 — 배터리 기여 적분 vs 1D THR
    if d is not None and "HRR_BATT" in d:
        thr_batt = _trapz(d["HRR_BATT"], d["Time"]) / 1e3    # [MJ]
        m = t1d <= d["Time"].max()
        ref = thr_1d[m][-1]
        rows.append(("#1 에너지 보존", f"FDS 배터리 THR {thr_batt:.1f} MJ / 1D {ref:.1f} MJ",
                     f"{thr_batt/max(ref,1e-9)*100:.1f} %",
                     "OK" if 0.9 <= thr_batt/max(ref,1e-9) <= 1.05 else "확인 필요 (미연소분/환기)"))
    # #2 케이싱 열유속
    if d is not None:
        qc = [f"q_{f}" for f in PACK_FACES if f"q_{f}" in d]
        if qc:
            q_fds = float(d[qc].mean(axis=1).max()); q_1d = float(CAS1D.q_net_kW_m2.max())
            rows.append(("#2 케이싱 열유속", f"FDS {q_fds:.2f} vs 1D 가정 {q_1d:.2f} kW/m²",
                         f"{q_fds/max(q_1d,1e-9):.2f}x",
                         "OK" if q_fds < 2 * q_1d else "1D 재실행 필요 (5.2)"))
    # #3/#4
    thr = _trapz(h["HRR"], h["Time"]) / 1e6
    # 계획된 T_END까지 못 갔으면 THR/피크는 아직 판정 대상이 아니다.
    t_sim = float(h["Time"].max())
    fds = c["dir"] / f"{MAIN}.fds"
    t_plan = t_sim
    if fds.exists():
        for line in fds.read_text().splitlines():
            if "T_END=" in line:
                t_plan = float(line.split("T_END=")[1].split()[0].rstrip("/,")); break
    # 계획 T_END에 못 미쳤거나, 애초에 브리프 기준(900 s)보다 짧은 케이스면 미판정
    partial = (t_sim < 0.98 * t_plan) or (t_plan < 0.98 * 900.0)
    tag = f"  (t={t_sim:.0f}/{t_plan:.0f} s)"
    rows.append(("#3 HRR 피크",
                 f"{h['HRR'].max():.0f} kW @ {h['Time'][h['HRR'].idxmax()]:.0f} s{tag}", "-",
                 "부분 실행 — 미판정" if partial else "Kang et al. Fig.6과 육안 대조"))
    rows.append(("#4 총 THR", f"{thr:.2f} GJ{tag}", "목표 7~8 GJ",
                 "부분 실행 — 미판정" if partial
                 else ("OK" if 6.5 <= thr <= 8.5 else "THICKNESS 재튜닝")))
    # #7
    if d is not None and "BATTGAS_underbody" in d:
        mx = float(d["BATTGAS_underbody"].max())
        rows.append(("#7 미연소 가스", f"하부 최대 Y={mx:.4f}", "-",
                     "OK (개방조건)" if mx < 0.05 else "축적 있음 — 환기조건 확인"))
    rows.append(("#5 격자 민감도", "sens_grid.png", "-", "OK" if T5 is not None else "미실행"))
    rows.append(("#6 복사분율 민감도", "sens_chi_r.png", "-", "OK" if T6 is not None else "미실행"))

    print(pd.DataFrame(rows, columns=["항목", "값", "비", "판정"]).to_string(index=False))


verdict()
print(f"\n그림: {FIGS}")
print(f"1D 되먹임 CSV: {FEEDBACK}")